In [28]:
import numpy as np
import os
from sklearn.model_selection import KFold, train_test_split

In [29]:
bin_size = 20
data_dir = "/burg/stats/users/yz4123/Downloads/nlb-rtt"
save_dir = "/burg/stats/users/yz4123/Downloads/nlb-rtt/xval"
os.makedirs(save_dir, exist_ok=True)

all_spikes = []; all_finger_vel = []
for split in ["train", "val", "test"]:
    all_behavior_means[split] = {}
    dataset = np.load(
        f"{data_dir}/{split}_dataset_{bin_size}.npy", allow_pickle=True
    ).item()
    all_spikes.append(dataset["spikes"])
    all_finger_vel.append(dataset["finger_vel"])
all_spikes = np.concatenate(all_spikes, axis=0)
all_finger_vel = np.concatenate(all_finger_vel, axis=0)

In [30]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (trainval_idx, test_idx) in enumerate(kf.split(all_spikes)):
    # Split remaining 80% into 70% train and 10% val (i.e., 7:1 ratio)
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=0.125, random_state=fold, shuffle=True
    )

    fold_dir = os.path.join(save_dir, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)

    # Save the splits
    np.save(os.path.join(fold_dir, f"train_dataset_{bin_size}.npy"), {
        "spikes": all_spikes[train_idx],
        "finger_vel": all_finger_vel[train_idx],
    })
    np.save(os.path.join(fold_dir, f"val_dataset_{bin_size}.npy"), {
        "spikes": all_spikes[val_idx],
        "finger_vel": all_finger_vel[val_idx],
    })
    np.save(os.path.join(fold_dir, f"test_dataset_{bin_size}.npy"), {
        "spikes": all_spikes[test_idx],
        "finger_vel": all_finger_vel[test_idx],
    })
    
print("Saved 5-fold data to:", save_dir)

Saved 5-fold data to: /burg/stats/users/yz4123/Downloads/nlb-rtt/xval
